# Transforms
> Batch transforms for spectral DataLoaders

In [ ]:
#| default_exp transforms

In [ ]:
#| export
import torch
from fastcore.all import *
from fasttransform import Transform
from soilspecdl.dataloader import TensorSpectrum

Spectra preprocessed for classical models (resampling, SNV, Savitzky-Golay derivative) can end up with very small magnitudes: SNV rescales each spectrum to unit variance, but a derivative taken afterwards shrinks it again. That scale suits PLSR or a random forest, but it sits well below what a CNN's default weight initialisation and `fastai.callback.hook.ActivationStats` diagnostics are tuned for, both of which assume roughly unit-scale input. 

`StandardizeSpectrum` rescales each spectrum independently, back to zero mean and unit variance across wavenumbers, right where it enters the network. It needs no fitted statistics, so it is exactly as valid on a batch of one as on the full training set, and it leaves the upstream preprocessing and the train/test split untouched.

In [ ]:
#| export
class StandardizeSpectrum(Transform):
    "Standardize each spectrum to zero mean, unit variance across wavenumbers"
    def __init__(self, eps=1e-6): store_attr()
    def encodes(self, x:TensorSpectrum):
        return (x - x.mean(-1, keepdim=True)) / (x.std(-1, keepdim=True) + self.eps)

For instance:

In [ ]:
#| eval: false
x = TensorSpectrum(torch.randn(32, 1, 657) * 0.02)
tfm = StandardizeSpectrum()
out = tfm(x)
out.mean(-1).abs().max(), out.std(-1).min(), out.std(-1).max()

(TensorSpectrum(shape=()), TensorSpectrum(shape=()), TensorSpectrum(shape=()))